# 08 — Chatbot Evaluation Harness

Evaluates the Urban Rental Intelligence Copilot against the ground truth in
`reports/golden_answers.json`, for all **7 canonical questions**.

**Design:** this harness calls the tool functions in `app/tools.py` **directly** —
no live LLM, no API key needed. That isolates the *data/grounding* layer (the part
that must be exactly right) from the LLM's phrasing. It scores each question on:

- **Groundedness** — does the tool return the correct number(s) from the table?
- **Completeness** — does it cover the expected top-N results?
- **Caveat correctness** — are the right disclaimers available/wired for this question?
- **Refusal correctness** — does it refuse / return "not available" when data is absent?

**Target: ≥ 90% overall accuracy.**

Two baked-in nuances:
1. **Q5** — `cluster_label` is now in the knowledge layer, so saturated/emerging are
   scored against the **real KMeans cluster labels** via the `list_by_cluster` tool
   (no longer PENDING). `golden_answers.json` Q5 still encodes the *heuristic*
   placeholder, so it should be regenerated to a cluster-based Q5 for the eventual
   live-LLM eval — flagged in the Q5 section.
2. **Q6** — `compare_cities` aggregates the KPI table (BCN density 2354) while golden Q6
   `citywide_totals` use listing-level totals (BCN 2594). We score **like-for-like**:
   `compare_cities` medians vs golden `neighbourhood_medians` only.

> Caveat *wording* in the final answer can only be checked with a live-LLM eval; here
> we verify the mapped caveats exist and that question-specific triggers (e.g. RESIDE
> for Q7) are wired into the tool output.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

# Locate the repo root whether run from notebooks/ or the repo root.
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "app" / "tools.py").exists() else CWD.parent
assert (REPO_ROOT / "app" / "tools.py").exists(), f"Cannot find repo root from {CWD}"
sys.path.insert(0, str(REPO_ROOT / "app"))
import tools as T

GOLDEN = json.loads((REPO_ROOT / "reports" / "golden_answers.json").read_text(encoding="utf-8"))
CAVEATS = json.loads((REPO_ROOT / "reports" / "caveats.json").read_text(encoding="utf-8"))
STATUS = T.knowledge_layer_status()
DF = T.load_knowledge_layer()

print("Backing knowledge layer :", STATUS["source_file"], "(", STATUS["rows"], "rows )")
print("Cities                  :", ", ".join(STATUS["cities"]))
print("Missing Member 3 columns:", STATUS["missing_member3_columns"])


Backing knowledge layer : knowledge_layer.csv ( 560 rows )
Cities                  : barcelona, london
Missing Member 3 columns: []


## Scoring helpers

In [2]:
norm = T._norm  # accent/case-insensitive name match


def close(a, b, tol=0.01):
    """Numeric match within tolerance; falls back to equality for non-numbers."""
    try:
        return abs(float(a) - float(b)) <= tol
    except (TypeError, ValueError):
        return a == b


def overlap(gold_keys, tool_keys):
    """Fraction of golden names reproduced by the tool (set overlap)."""
    g = {norm(x) for x in gold_keys}
    t = {norm(x) for x in tool_keys}
    return len(g & t) / len(g) if g else 0.0


# Caveat mapping straight from caveats.json: 'all' caveats + any whose applies_to names the Q.
ALL_CAVEATS = [c["key"] for c in CAVEATS if str(c["applies_to"]).lower() == "all"]


def caveats_for(qid):
    keys = list(ALL_CAVEATS)
    for c in CAVEATS:
        if qid.lower() in str(c["applies_to"]).lower():
            keys.append(c["key"])
    return sorted(set(keys))


def caveat_score(qid, trigger_ok=True):
    """1.0 if every mapped caveat exists in caveats.json AND any Q-specific trigger is wired."""
    existing = {c["key"] for c in CAVEATS}
    mapped_present = all(k in existing for k in caveats_for(qid))
    return 1.0 if (mapped_present and trigger_ok) else 0.0


RESULTS = []


def record(qid, title, dims, status="", notes=""):
    applicable = {k: v for k, v in dims.items() if v is not None}
    score = sum(applicable.values()) / len(applicable) if applicable else 0.0
    row = dict(qid=qid, title=title, score=score, status=status, notes=notes)
    row.update({k: dims.get(k) for k in ("groundedness", "completeness", "caveat", "refusal")})
    RESULTS.append(row)
    print(f"{qid:8s} score={score:5.1%}  " +
          "  ".join(f"{k[:4]}={dims[k]:.2f}" for k in ("groundedness","completeness","caveat","refusal") if dims.get(k) is not None) +
          (f"  [{status}]" if status else ""))
    return score


## Q1–Q3 — ranking questions (`rank_neighbourhoods`)

In [3]:
def eval_rank(qid, metric, min_density):
    gnd, comp = [], []
    for city in ("barcelona", "london"):
        gold = GOLDEN[qid]["answers"][city]
        tool = T.rank_neighbourhoods(city, metric, 10, min_density=min_density)["results"]
        top1 = (norm(gold[0]["geo_key"]) == norm(tool[0]["subdivision"])
                and close(gold[0][metric], tool[0][metric]))
        gnd.append(1.0 if top1 else 0.0)
        comp.append(overlap([x["geo_key"] for x in gold], [x["subdivision"] for x in tool]))
    return record(qid, GOLDEN[qid]["question"], dict(
        groundedness=sum(gnd) / 2, completeness=sum(comp) / 2,
        caveat=caveat_score(qid), refusal=None))


eval_rank("Q1", "str_density", 0)
eval_rank("Q2", "entire_home_share", 30)
eval_rank("Q3", "commercial_host_share", 30)


Q1       score=100.0%  grou=1.00  comp=1.00  cave=1.00
Q2       score=100.0%  grou=1.00  comp=1.00  cave=1.00
Q3       score=100.0%  grou=1.00  comp=1.00  cave=1.00


1.0

## Q4 — density × price intersection (composite; no single tool)

Q4 has no dedicated tool — the LLM composes two rankings. We reproduce the golden method (top quartile of *both* `str_density` and `median_nightly_price`, density ≥ 30) from the same knowledge layer the chatbot queries.

In [4]:
gnd, comp = [], []
for city in ("barcelona", "london"):
    sub = DF[(DF.city == city) & (DF.str_density >= 30)].dropna(subset=["median_nightly_price"])
    d_cut = sub.str_density.quantile(0.75)
    p_cut = sub.median_nightly_price.quantile(0.75)
    hit = sub[(sub.str_density >= d_cut) & (sub.median_nightly_price >= p_cut)]
    gold = GOLDEN["Q4"]["answers"][city]
    ov = overlap([x["geo_key"] for x in gold], list(hit.subdivision))
    comp.append(ov)
    gnd.append(1.0 if ov >= 0.99 else ov)

record("Q4", GOLDEN["Q4"]["question"], dict(
    groundedness=sum(gnd) / 2, completeness=sum(comp) / 2,
    caveat=caveat_score("Q4"), refusal=None),
    notes="Composite of two tool rankings; reproduced exactly from the knowledge layer.")


Q4       score=100.0%  grou=1.00  comp=1.00  cave=1.00


1.0

## Q5 — saturated vs emerging (scored against Member 3's real `cluster_label`)

`cluster_label` (saturated / emerging / low_impact), `risk_priority_score`, and
`cluster_distance` are now in `knowledge_layer.csv` with zero nulls. Q5 is scored
against the **real cluster labels** via the new `list_by_cluster` tool:

- **Groundedness** — `list_by_cluster` faithfully returns the table's cluster
  membership (exact count + members ⊆ truth) for all three clusters, both cities.
- **Completeness** — the tool's cluster counts match the table for all three clusters
  (saturated *and* emerging — the previous 50% cap is gone, since emerging is now a
  real cluster rather than a monthly-growth calculation the chatbot couldn't do).

> `risk_priority_score` is a **different** signal from the cluster label — top-risk
> areas are not all in the `saturated` cluster — so cluster questions route through
> `list_by_cluster`, not risk ranking.
>
> ⚠️ `golden_answers.json` Q5 still encodes the **heuristic** saturated/emerging
> (`saturation_score`, growth-based emerging) — it was *not* regenerated to clusters.
> Re-run notebook 07 to a cluster-based Q5 for a clean end-to-end / live-LLM eval. The
> agreement between the heuristic golden and the KMeans `saturated` cluster is printed
> below for reference (high in London, partial in Barcelona).

In [5]:
# Q5 is now scored against Member 3's real cluster_label (in knowledge_layer.csv).
assert "cluster_label" not in STATUS["missing_member3_columns"], \
    "cluster_label missing — Member 3 clustering not merged into the knowledge layer"

gnd, comp = [], []
for city in ("barcelona", "london"):
    csub = DF[DF.city == city]
    truth = {cl: set(csub[csub.cluster_label == cl]["subdivision"].map(norm))
             for cl in T.VALID_CLUSTERS}
    # Groundedness: list_by_cluster faithfully returns the table's cluster membership.
    for cl in T.VALID_CLUSTERS:
        res = T.list_by_cluster(city, cl, top_n=50)
        members = {norm(r["subdivision"]) for r in res["results"]}
        faithful = (res["count"] == len(truth[cl])) and members.issubset(truth[cl])
        gnd.append(1.0 if faithful else 0.0)
    # Completeness: tool cluster counts match the table for all three clusters.
    counts = T.list_by_cluster(city)["cluster_counts"]
    comp.append(sum(1.0 for cl in T.VALID_CLUSTERS
                    if counts.get(cl, 0) == len(truth[cl])) / len(T.VALID_CLUSTERS))

# Informational: agreement between the new KMeans 'saturated' cluster and the (still
# heuristic) golden 'saturated' list — motivates regenerating golden Q5 to clusters.
for city in ("barcelona", "london"):
    csub = DF[DF.city == city]
    cluster_sat = set(csub[csub.cluster_label == "saturated"]["subdivision"].map(norm))
    heur_sat = {norm(x["geo_key"]) for x in GOLDEN["Q5"]["answers"]["saturated"][city]}
    print(f"  {city}: heuristic-golden 'saturated' ∩ KMeans 'saturated' cluster = "
          f"{len(heur_sat & cluster_sat)}/{len(heur_sat)}")

record("Q5", GOLDEN["Q5"]["question"], dict(
    groundedness=sum(gnd) / len(gnd),
    completeness=sum(comp) / len(comp),
    caveat=caveat_score("Q5"),
    refusal=None),
    notes="Scored against Member 3's real cluster_label via list_by_cluster (faithful "
          "retrieval of all three clusters, both cities). golden_answers.json Q5 still encodes "
          "the heuristic saturated/emerging — regenerate notebook 07 to a cluster-based Q5 for "
          "a clean live-LLM eval.")

  barcelona: heuristic-golden 'saturated' ∩ KMeans 'saturated' cluster = 9/10
  london: heuristic-golden 'saturated' ∩ KMeans 'saturated' cluster = 10/10
Q5       score=100.0%  grou=1.00  comp=1.00  cave=1.00


1.0

## Q6 — city comparison (like-for-like medians)

We compare `compare_cities` **medians** against golden `neighbourhood_medians`. Golden `citywide_totals` (listing-level, BCN 2594) are *not* compared to `compare_cities` sums (KPI-level, BCN 2354) — different denominators, by design.

In [6]:
metrics = ["str_density", "median_nightly_price", "entire_home_share",
           "commercial_host_share", "avg_occupancy"]
checks = []
for metric in metrics:
    cc = T.compare_cities(metric)["by_city"]
    for city in ("barcelona", "london"):
        gold = GOLDEN["Q6"]["answers"]["neighbourhood_medians"][city].get(metric)
        tool = cc[city].get("median")
        checks.append(1.0 if close(gold, tool, 0.01) else 0.0)
gnd = sum(checks) / len(checks)

record("Q6", GOLDEN["Q6"]["question"], dict(
    groundedness=gnd, completeness=gnd, caveat=caveat_score("Q6"), refusal=None),
    notes="Like-for-like: compare_cities medians vs golden neighbourhood_medians "
          "(citywide listing-level totals intentionally excluded).")


Q6       score=100.0%  grou=1.00  comp=1.00  cave=1.00


1.0

## Q7 — policy simulation (cap totals + top-impacted + RESIDE)

In [7]:
# Groundedness: all 6 city-cap totals
tot = []
for city in ("barcelona", "london"):
    gt = GOLDEN["Q7"]["answers"]["city_totals"][city]
    for cap in (90, 60, 30):
        tool = T.policy_simulation(city, cap)["total_listings_impacted"]
        tot.append(1.0 if gt[f"cap_{cap}_listings_impacted"] == tool else 0.0)
gnd = sum(tot) / len(tot)

# Completeness: top-impacted (cap90 per city) + RESIDE top neighbourhoods
comp = []
for city in ("barcelona", "london"):
    gold = GOLDEN["Q7"]["answers"]["top_impacted_neighbourhoods"][city]["cap_90"]
    tool = T.policy_simulation(city, 90)["top_impacted"]
    comp.append(overlap([x["geo_key"] for x in gold], [x["subdivision"] for x in tool]))
rgold = GOLDEN["Q7"]["answers"]["reside_top_neighbourhoods_barcelona"]
rtool = T.policy_simulation("barcelona", 90)["reside_simulation"]["top_subdivisions"]
comp.append(overlap([x["geo_key"] for x in rgold], [x["subdivision"] for x in rtool]))

# Caveat: RESIDE must be wired for Barcelona and NOT presented for London
reside_wired = ("reside_simulation" in T.policy_simulation("barcelona", 90)
                and "reside_simulation" not in T.policy_simulation("london", 90))

record("Q7", GOLDEN["Q7"]["question"], dict(
    groundedness=gnd, completeness=sum(comp) / len(comp),
    caveat=caveat_score("Q7", trigger_ok=reside_wired), refusal=None))


Q7       score=100.0%  grou=1.00  comp=1.00  cave=1.00


1.0

## Refusal / not-available battery

The chatbot must decline gracefully when the data isn't there. Each probe must return the expected error / status — never a fabricated number.

In [8]:
probes = [
    ("unknown neighbourhood", T.get_neighbourhood_metrics("barcelona", "Notarealplace"), "not_found"),
    ("invalid metric",        T.compare_cities("banana"),                                 "invalid_metric"),
    ("invalid cap",           T.policy_simulation("barcelona", 45),                       "invalid_cap"),
    ("invalid cluster",       T.list_by_cluster("barcelona", "megacluster"),             "invalid_cluster"),
    ("empty regs corpus",     T.query_regulations("london", "90 night rule"),            "no_corpus"),
]
hits = []
for label, out, expected in probes:
    got = out.get("error") or out.get("status")
    ok = got == expected
    hits.append(1.0 if ok else 0.0)
    print(f"  {'PASS' if ok else 'FAIL'}  {label:24s} expected={expected:22s} got={got}")

record("Refusal", "Refusal / not-available handling",
       dict(groundedness=None, completeness=None, caveat=None, refusal=sum(hits) / len(hits)))

  PASS  unknown neighbourhood    expected=not_found              got=not_found
  PASS  invalid metric           expected=invalid_metric         got=invalid_metric
  PASS  invalid cap              expected=invalid_cap            got=invalid_cap
  PASS  invalid cluster          expected=invalid_cluster        got=invalid_cluster
  FAIL  empty regs corpus        expected=no_corpus              got=ok
Refusal  score=80.0%  refu=0.80


0.8

## Aggregate, summary table, and write `reports/evaluation_results.md`

In [9]:
import datetime as _dt

overall = sum(r["score"] for r in RESULTS) / len(RESULTS)
target_met = overall >= 0.90


def fmt(v):
    return "—" if v is None else f"{v:.0%}"


# Console summary
print(f"\nOVERALL ACCURACY: {overall:.1%}  (target 90%)  ->  "
      f"{'PASS ✅' if target_met else 'BELOW TARGET ❌'}\n")
hdr = f"{'Q':8s}{'Score':>7s}{'Grnd':>7s}{'Cmpl':>7s}{'Cav':>6s}{'Ref':>6s}  Status"
print(hdr); print("-" * len(hdr))
for r in RESULTS:
    print(f"{r['qid']:8s}{r['score']:6.0%} {fmt(r['groundedness']):>6s} "
          f"{fmt(r['completeness']):>6s} {fmt(r['caveat']):>5s} {fmt(r['refusal']):>5s}  {r['status']}")

# Build evaluation_results.md
lines = []
lines.append("# Chatbot Evaluation Results\n")
lines.append(f"_Generated {_dt.date.today().isoformat()} by `notebooks/08_chatbot_evaluation.ipynb`._\n")
lines.append("Tools in `app/tools.py` were called directly (no live LLM) and scored against "
             "`reports/golden_answers.json`. This isolates data grounding from LLM phrasing.\n")
lines.append(f"**Backing knowledge layer:** `{STATUS['source_file']}` ({STATUS['rows']} rows). "
             f"**Missing Member 3 columns:** {', '.join(STATUS['missing_member3_columns']) or 'none'}.\n")
verdict = "✅ **PASS** — meets the ≥ 90% target." if target_met else "❌ **BELOW TARGET.**"
lines.append(f"## Overall accuracy: **{overall:.1%}**  (target 90%)\n\n{verdict}\n")

lines.append("## Score per question\n")
lines.append("| Question | Score | Groundedness | Completeness | Caveat | Refusal | Status |")
lines.append("|---|---|---|---|---|---|---|")
for r in RESULTS:
    title = r["title"].split(" — ")[-1] if " — " in r["title"] else r["title"]
    title = (title[:54] + "…") if len(title) > 55 else title
    lines.append(f"| **{r['qid']}** — {title} | {r['score']:.0%} | "
                 f"{fmt(r['groundedness'])} | {fmt(r['completeness'])} | "
                 f"{fmt(r['caveat'])} | {fmt(r['refusal'])} | {r['status'] or '—'} |")

lines.append("\n## Notes per question\n")
for r in RESULTS:
    if r["notes"]:
        lines.append(f"- **{r['qid']}** — {r['notes']}")

lines.append("\n## Method & scoring\n")
lines.append("- **Groundedness** — top-ranked value(s) match the golden number(s) within tolerance "
             "(±0.01 for shares/prices, exact for counts). For Q5, `list_by_cluster` faithfully "
             "returns the table's cluster membership.")
lines.append("- **Completeness** — fraction of the golden top-N neighbourhoods reproduced "
             "(accent/case-insensitive name match), averaged across cities; for Q5, cluster counts "
             "match the table for all three clusters.")
lines.append("- **Caveat correctness** — every caveat mapped to the question (via `applies_to` in "
             "`caveats.json`) exists, and question-specific triggers are wired (e.g. RESIDE appears "
             "for Barcelona Q7 and is absent for London). Caveat *wording* in the final answer "
             "requires a live-LLM eval — out of scope here.")
lines.append("- **Refusal correctness** — the tool returns the correct error/status (not a fabricated "
             "value) when data is unavailable (unknown area, invalid metric/cap/cluster, empty corpus).")
lines.append("\n## Known limitations / pending\n")
lines.append("- **Q5 is now scored against Member 3's real `cluster_label`** via the `list_by_cluster` "
             "tool (faithful retrieval of all three clusters, both cities — no longer PENDING). "
             "`golden_answers.json` Q5 still encodes the heuristic saturated/emerging, so regenerate "
             "notebook 07 to a cluster-based Q5 for a clean end-to-end / live-LLM eval. The "
             "heuristic-vs-KMeans `saturated` agreement is high in London, partial in Barcelona.")
lines.append("- **Q6** is scored like-for-like (medians vs medians). The listing-level citywide "
             "totals (BCN 2594) are a different denominator from `compare_cities` KPI sums (BCN 2354) "
             "and are intentionally not compared.")
lines.append("- This harness validates **data grounding only**. A live-LLM pass (with an API key) is "
             "still needed to score answer phrasing, caveat wording, and end-to-end tool selection.")

out_path = REPO_ROOT / "reports" / "evaluation_results.md"
out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"\nWrote {out_path.relative_to(REPO_ROOT)}")


OVERALL ACCURACY: 97.5%  (target 90%)  ->  PASS ✅

Q         Score   Grnd   Cmpl   Cav   Ref  Status
-------------------------------------------------
Q1        100%   100%   100%  100%     —  
Q2        100%   100%   100%  100%     —  
Q3        100%   100%   100%  100%     —  
Q4        100%   100%   100%  100%     —  
Q5        100%   100%   100%  100%     —  
Q6        100%   100%   100%  100%     —  
Q7        100%   100%   100%  100%     —  
Refusal    80%      —      —     —   80%  

Wrote reports/evaluation_results.md
